# Dataset Audit v1 — `dataset2`

**Purpose.** This notebook documents the first quality audit of `dataset2`, the raw corpus used to train the PII
detector of the Contextual PII Anonymizer. It answers five questions that must be settled before any transformation,
model selection, or training: what the dataset contains, how consistent its annotations are, which classes are
under-represented, whether duplicated or leaked documents exist, and which limitations constrain training.

**Audience.** The thesis director and any contributor who needs to approve the transition from raw data to
preparation. Every section is self-contained: it states what is measured, why it matters, how to read the result,
and which conclusion follows. No external explanation is required to review this document.

**Dataset identity.** `dataset2` is versioned with DVC and restored with `dvc pull`. The committed metadata fixes the
content: `md5 5d32321fd6706a62976cba08dff97278.dir`, 10 files, 311,156,761 bytes. Git history therefore identifies the
exact version this notebook analysed.

**How to run.** From the `ml` directory:

```powershell
uv sync
uv run jupyter lab
```

Open this notebook with the `ml` environment kernel and run all cells. The notebook only reads data. It writes
figures to `datasets/dataset2/reports/figures/` and never modifies the dataset.

**Definitions used throughout.**

- **Document**: one record of `dataset.jsonl`, with free text and the PII values declared by its generator.
- **Example**: one word-level record of `etiquetado/*.jsonl`, with `tokens` and BIO `ner_tags`.
- **BIO tag**: `B-X` opens entity type `X`, `I-X` continues it, `O` marks a token outside any entity.
- **Entity**: a maximal run of tokens that belongs to one entity type.
- **Split**: one partition of the labelled data (`train`, `val`, `test`).
- **Coverage**: the share of documents that contain at least one entity of a given class.

**Scope.** This is a descriptive audit of the frozen version. It does not repair labels, remove records, or train a
model. Identified problems are recorded in `reports/dataset_audit_v1.md` so that preparation decisions are traceable.


In [1]:
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import openpyxl
import pandas as pd

from pii_ml.data import audit, validation

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 170)
pd.set_option("display.max_rows", 120)


def find_workspace() -> Path:
    """Locate the ml workspace from the notebook directory."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "data" / "raw" / "dataset2").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from inside the ml workspace")


ML_ROOT = find_workspace()
DATASET_ROOT = ML_ROOT / "data" / "raw" / "dataset2"
REPORTS_DIR = ML_ROOT / "datasets" / "dataset2" / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_figure(figure: plt.Figure, name: str) -> None:
    path = FIGURES_DIR / name
    figure.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(figure)
    print(f"Saved {path.relative_to(ML_ROOT).as_posix()}")


print(f"Python {platform.python_version()} | pandas {pd.__version__}")
print(f"Dataset root: {DATASET_ROOT.relative_to(ML_ROOT).as_posix()}")
print(f"Reports dir:  {REPORTS_DIR.relative_to(ML_ROOT).as_posix()}")

Python 3.14.8 | pandas 2.3.3
Dataset root: data/raw/dataset2
Reports dir:  datasets/dataset2/reports


## 1. Inventory

**What.** Enumerate every file delivered inside the dataset directory, with its size and, for text artifacts, its
line count. A dataset whose contents are unknown cannot be audited: the inventory establishes the exact payload that
the DVC version describes and exposes artifacts that carry no analytical value.

**Why.** File counts and sizes must agree with the DVC metadata (10 files, ~311 MB). A mismatch means the working
copy is not the frozen version. An empty or duplicated artifact inflates the dataset without adding information.


In [2]:
inventory = audit.file_inventory(DATASET_ROOT)
inventory_frame = pd.DataFrame(
    [
        {
            "path": entry.path,
            "size_mib": round(entry.size_bytes / (1024 * 1024), 2),
            "lines": entry.lines,
            "format": entry.suffix,
        }
        for entry in inventory
    ]
)
display(inventory_frame)
print(
    f"{len(inventory_frame)} files | "
    f"{inventory_frame['size_mib'].sum():.2f} MiB | "
    f"{inventory_frame['lines'].sum()} text lines"
)

,path,size_mib,lines,format
0,dataset.jsonl,60.31,16144.0,.jsonl
1,dataset.xlsx,18.59,NaN,.xlsx
2,etiquetado/labels.txt,0.00,63.0,.txt
3,etiquetado/test.conll,8.60,959797.0,.conll
4,etiquetado/test.jsonl,13.29,1615.0,.jsonl
5,etiquetado/train.conll,68.38,7638564.0,.conll
6,etiquetado/train.jsonl,105.70,12915.0,.jsonl
7,etiquetado/val.conll,8.59,958439.0,.conll
8,etiquetado/val.jsonl,13.27,1614.0,.jsonl
9,prueba.json,0.00,0.0,.json


10 files | 296.73 MiB | 9589151.0 text lines


**What to look for.** The DVC metadata declares 10 files and 311,156,761 bytes; the inventory must match both.
`prueba.json` is inspected below because a zero-byte JSON artifact cannot be parsed and therefore cannot belong to a
reproducible dataset version.


In [3]:
empty_files = [entry.path for entry in inventory if entry.size_bytes == 0]
print(f"Empty files: {empty_files or 'none'}")

workbook = openpyxl.load_workbook(DATASET_ROOT / "dataset.xlsx", read_only=True)
sheet = workbook["dataset"]
header = next(sheet.iter_rows(min_row=1, max_row=1, values_only=True))
print(f"Workbook sheets: {workbook.sheetnames}")
print(f"Sheet 'dataset': {sheet.max_row} rows (header included) x {sheet.max_column} columns")
print(f"Columns: {list(header)}")
workbook.close()

Empty files: ['prueba.json']
Workbook sheets: ['dataset']
Sheet 'dataset': 16145 rows (header included) x 6 columns
Columns: ['id', 'tipo_documento', 'texto', 'num_palabras', 'num_elementos_pii', 'elementos_taxonomia_utilizados']


### Reading the inventory

The payload is one tabular export (`dataset.xlsx`), one JSONL export of the same corpus (`dataset.jsonl`), the
labelled partitions in two redundant encodings (`*.jsonl` and `*.conll`), the label vocabulary, and one empty file.
The `.xlsx` export is a third representation of the same 16,144 documents (16,145 rows including the header), so it
adds shipping weight but no new information; it is useful only as a human-readable sample and must never become an
independent source of truth.


## 2. Schema

**What.** Record the fields, runtime types, and presence of every record in `dataset.jsonl` and in the labelled
splits. This defines the contract that preparation code is allowed to depend on.

**Why.** Silent schema variation (a field present in 90% of records, a numeric field stored as text) breaks
transformations far from the point where the defect was introduced.


In [4]:
document_schema = audit.schema_report(audit.iter_jsonl(DATASET_ROOT / "dataset.jsonl"))
schema_frame = pd.DataFrame(
    [
        {
            "field": field,
            "present_in": values["count"],
            "coverage": f"{values['coverage']:.2%}",
            "python_types": ", ".join(sorted(values["types"])),
        }
        for field, values in document_schema.items()
    ]
)
display(schema_frame)

sample = next(audit.iter_jsonl(DATASET_ROOT / "dataset.jsonl"))
print("\nExample document (text and PII values truncated):")
print(
    {
        "id": sample["id"],
        "tipo_documento": sample["tipo_documento"],
        "pii_incluida": sample["pii_incluida"],
        "num_palabras": sample["num_palabras"],
        "cobertura_pii": sample["cobertura_pii"],
        "texto": sample["texto"][:120] + "...",
    }
)

,field,present_in,coverage,python_types
0,id,16144,100.00%,int
1,tipo_documento,16144,100.00%,str
2,pii_incluida,16144,100.00%,list
3,valores_pii,16144,100.00%,dict
4,texto,16144,100.00%,str
5,num_palabras,16144,100.00%,int
6,cobertura_pii,16144,100.00%,float
7,pii_faltante,16144,100.00%,list



Example document (text and PII values truncated):
{'id': 0, 'tipo_documento': 'contrato_laboral', 'pii_incluida': ['nombres', 'apellidos', 'cedula', 'empresa', 'profesion', 'correo_electronico', 'salario', 'cargo'], 'num_palabras': 505, 'cobertura_pii': 1.0, 'texto': 'En la ciudad de Quito, Distrito Metropolitano, se celebra el presente contrato individual de trabajo entre la entidad ju...'}


In [5]:
example_schema = audit.schema_report(audit.iter_jsonl(DATASET_ROOT / "etiquetado" / "train.jsonl"))
display(
    pd.DataFrame(
        [
            {
                "field": field,
                "present_in": values["count"],
                "coverage": f"{values['coverage']:.2%}",
                "python_types": ", ".join(sorted(values["types"])),
            }
            for field, values in example_schema.items()
        ]
    )
)

example = next(audit.iter_jsonl(DATASET_ROOT / "etiquetado" / "train.jsonl"))
print("\nExample BIO record:")
print(
    {
        "id": example["id"],
        "tipo_documento": example["tipo_documento"],
        "tokens": len(example["tokens"]),
        "ner_tags": example["ner_tags"][:12],
        "tokens_sample": example["tokens"][:12],
    }
)

,field,present_in,coverage,python_types
0,id,12915,100.00%,int
1,tipo_documento,12915,100.00%,str
2,tokens,12915,100.00%,list
3,ner_tags,12915,100.00%,list



Example BIO record:
{'id': 2985, 'tipo_documento': 'historia_clinica', 'tokens': 690, 'ner_tags': ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'], 'tokens_sample': ['Se', 'registra', 'la', 'atención', 'médica', 'en', 'el', 'servicio', 'de', 'consulta', 'externa', 'de']}


### Reading the schema

`dataset.jsonl` carries the generation contract: the document type, the declared PII fields, the concrete values used
to build the text, the word count, and a declared coverage score. The labelled splits carry only what a token
classifier needs: identifier, document type, tokens, and BIO tags. The two artifacts are joined by `id`.

Because the raw artifact declares both `pii_incluida` and `valores_pii`, label quality can be audited against an
independent declaration instead of only against the tags themselves. That is the strongest verification available
for this dataset and it is used in Section 8.


## 3. Integrity

**What.** Run the structural validator over the raw documents, the three labelled splits, and the cross-artifact
relations. Each check reports an aggregated issue with a severity.

**Why.** Integrity defects make a dataset unusable or silently misleading. The checks cover required fields, types,
unique identifiers, empty content, token/tag length agreement, BIO grammar, and split isolation.


In [6]:
issues, splits = validation.validate_dataset(DATASET_ROOT)
summary = validation.summarise(issues)

issues_frame = pd.DataFrame(
    [
        {
            "severity": issue.severity,
            "code": issue.code,
            "occurrences": issue.count,
            "message": issue.message,
            "sample": issue.samples[0] if issue.samples else "",
        }
        for issue in issues
    ]
)
display(issues_frame)
print(
    f"Total findings: {summary['total']} | errors: {summary['errors']} | warnings: {summary['warnings']}"
)

,severity,code,occurrences,message,sample
0,warning,tokenization_drift,1207,BIO tokens differ from punctuation-aware token...,id=2985 tokens=690 punctuation=686


Total findings: 1 | errors: 0 | warnings: 1


### Reading the integrity results

An empty findings table means the dataset satisfies every checked contract. A single warning about tokenization
means that in 1,207 of 16,144 documents (7.5%) the BIO token boundaries are not reproducible from the raw text by
punctuation-aware splitting; the labels remain valid, but preparation code must not re-tokenize from `texto` and
assume identical alignments.

BIO grammar is validated strictly: `I-X` must continue an open `X`, and every tag must belong to the declared
vocabulary. Section 8 quantifies the remaining annotation-quality signals.


## 4. Labels

**What.** Compare the declared label vocabulary with the classes that actually occur, then count entities and the
documents that contain them.

**Why.** A declared class with zero occurrences cannot be learned. A class that appears in very few documents cannot
generalize. The declared-versus-observed comparison separates design intent from usable evidence.


In [7]:
vocabulary = audit.read_label_vocabulary(DATASET_ROOT / "etiquetado" / "labels.txt")
declared_types = sorted(tag[len("B-") :] for tag in vocabulary if tag.startswith("B-"))

examples = [example for split in splits.values() for example in split]
entity_counts = audit.class_counts(examples)
document_counts = audit.class_document_counts(examples)
observed_types = sorted(entity_counts)

absent_types = sorted(set(declared_types) - set(entity_counts))
unexpected_types = sorted(set(entity_counts) - set(declared_types))

print(f"Vocabulary entries (including O): {len(vocabulary)}")
print(f"Declared entity types: {len(declared_types)}")
print(f"Observed entity types: {len(observed_types)}")
print(f"Declared but absent: {absent_types}")
print(f"Observed but undeclared: {unexpected_types}")
print(f"Total entities: {sum(entity_counts.values())}")

Vocabulary entries (including O): 63
Declared entity types: 31
Observed entity types: 29
Declared but absent: ['GENERO', 'HISTORIAL_CLINICO']
Observed but undeclared: []
Total entities: 154142


In [8]:
observed_tags = {tag for example in examples for tag in example.tags}
label_rows = []
for entity_type in sorted(entity_counts, key=lambda name: entity_counts[name], reverse=True):
    inside_tag = f"I-{entity_type}"
    label_rows.append(
        {
            "entity_type": entity_type,
            "entities": entity_counts[entity_type],
            "documents": document_counts[entity_type],
            "share_of_documents": document_counts[entity_type] / len(examples),
            "multi_token_observed": inside_tag in observed_tags,
        }
    )

label_frame = pd.DataFrame(label_rows)
label_frame["share_of_documents"] = label_frame["share_of_documents"].map(lambda value: f"{value:.1%}")
display(label_frame)
print(f"Entity types with at least one multi-token entity: {label_frame['multi_token_observed'].sum()}")

,entity_type,entities,documents,share_of_documents,multi_token_observed
0,NOMBRES,33281,16143,100.0%,True
1,APELLIDOS,30339,16144,100.0%,True
2,CEDULA,12880,10431,64.6%,False
3,EMPRESA,11325,5475,33.9%,True
4,CORREO_ELECTRONICO,8603,8004,49.6%,True
5,DIRECCION,6437,4954,30.7%,True
6,USUARIO,5149,4202,26.0%,True
7,CARGO,4817,3352,20.8%,True
8,NUMERO_TELEFONICO,4654,4447,27.5%,False
9,PLACA_VEHICULO,4257,2310,14.3%,True


Entity types with at least one multi-token entity: 18


In [9]:
figure, axis = plt.subplots(figsize=(9, 7))
ordered = entity_counts.most_common()
names = [name for name, _ in ordered]
values = [count for _, count in ordered]
axis.barh(names[::-1], values[::-1], color="#3b6ea5")
axis.set_xscale("log")
axis.set_xlabel("Entity occurrences (log scale)")
axis.set_title("Entity occurrences by class")
axis.grid(axis="x", alpha=0.3)
save_figure(figure, "002_class_distribution.png")

Saved datasets/dataset2/reports/figures/002_class_distribution.png


### Reading the labels

The vocabulary declares 31 entity types, each with a `B-` and an `I-` tag, plus `O`. Two declared types, `GENERO`
and `HISTORIAL_CLINICO`, never occur in the labelled data, so the trainable label space is 29 types. Eleven of those
29 types only ever appear as single tokens; the remaining 18 also occur as multi-token entities.

The distribution is long-tailed: names and surnames dominate, while `EDAD`, `ESTADO_CIVIL`, `PESO`, `ENFERMEDADES`
and `TALLA_ESTATURA` appear in a small fraction of documents. Section 9 converts these counts into coverage
constraints for training and evaluation.


## 5. Distributions

**What.** Describe the document types and the size of the documents in tokens, then quantify class imbalance.

**Why.** The model sees a mixture of document templates. If one template dominates, the model can fit its wording
instead of learning the PII patterns. Document length matters for truncation limits and for the share of examples
whose entities survive tokenization.


In [10]:
document_types = audit.document_type_counts(examples)
type_frame = (
    pd.DataFrame(
        [
            {"document_type": name, "documents": count, "share": count / len(examples)}
            for name, count in document_types.most_common()
        ]
    )
)
type_frame["share"] = type_frame["share"].map(lambda value: f"{value:.1%}")
display(type_frame)

figure, axis = plt.subplots(figsize=(9, 5))
axis.barh(
    [name for name, _ in document_types.most_common()][::-1],
    [count for _, count in document_types.most_common()][::-1],
    color="#5b8c5a",
)
axis.set_xlabel("Documents")
axis.set_title("Documents per document type")
axis.grid(axis="x", alpha=0.3)
save_figure(figure, "001_document_types.png")

,document_type,documents,share
0,denuncia_policial,2355,14.6%
1,contrato_laboral,2265,14.0%
2,factura_comercial,2203,13.6%
3,publicacion_red_social,2075,12.9%
4,reclamo_lopdp,2059,12.8%
5,reclamo_bancario,1840,11.4%
6,correo_soporte,1431,8.9%
7,formulario_matricula,1145,7.1%
8,historia_clinica,771,4.8%


Saved datasets/dataset2/reports/figures/001_document_types.png


In [11]:
token_lengths = audit.document_lengths(examples)
length_stats = audit.length_stats(token_lengths)
display(pd.DataFrame([length_stats]))

figure, axis = plt.subplots(figsize=(9, 4.5))
axis.hist(token_lengths, bins=40, color="#8a6bbf", edgecolor="white")
axis.axvline(length_stats["p99"], color="#b23a48", linestyle="--", label=f"p99 = {length_stats['p99']:.0f} tokens")
axis.set_xlabel("Tokens per document")
axis.set_ylabel("Documents")
axis.set_title("Document length distribution")
axis.legend()
axis.grid(axis="y", alpha=0.3)
save_figure(figure, "003_document_lengths.png")

,count,min,p50,p90,p99,max,mean
0,16144,115.0,592.0,700.0,765.0,984.0,590.97225


Saved datasets/dataset2/reports/figures/003_document_lengths.png


In [12]:
entities_per_document = [sum(1 for tag in example.tags if tag != "O") for example in examples]
ordered_counts = sorted(entity_counts.values())
imbalance = ordered_counts[-1] / ordered_counts[0]

print(f"Documents with entities: {len(examples) - audit.documents_without_entities(examples)} of {len(examples)}")
print(f"Entities per document: min={min(entities_per_document)} max={max(entities_per_document)} "
      f"mean={sum(entities_per_document) / len(entities_per_document):.2f}")
print(f"Class imbalance (largest / smallest): {imbalance:.1f}x")
print(f"Classes with fewer than 1000 entities: "
      f"{sum(1 for count in entity_counts.values() if count < 1000)} of {len(entity_counts)}")

Documents with entities: 16144 of 16144
Entities per document: min=5 max=91 mean=26.38
Class imbalance (largest / smallest): 60.0x
Classes with fewer than 1000 entities: 7 of 29


### Reading the distributions

Nine document types are present, with the largest (`denuncia_policial`) holding roughly one seventh of the corpus,
so no template dominates. Documents are long: the median is 592 tokens and the 99th percentile stays below
800, which means a 512-token truncation limit would cut more than half of the documents and lose their entity spans.

The class imbalance is severe: the most frequent class has 33,281 occurrences and the least frequent has 555, a
ratio of 60:1. A per-class metric is therefore mandatory during evaluation; accuracy or micro-averaged F1 would hide
the weak classes entirely.


## 6. Duplicates

**What.** Detect identical documents, documents identical after normalization (case, accents, whitespace), and
near-duplicates that share their opening sequence with a token-set Jaccard similarity of at least 0.9.

**Why.** Duplicated documents inflate metrics and, when they cross splits, leak evaluation data into training. Exact
duplicates are a defect; near-duplicates are a warning about template repetition.


In [13]:
documents = audit.load_documents(DATASET_ROOT / "dataset.jsonl")
duplicates = audit.duplicate_report(documents)

duplicate_frame = pd.DataFrame(
    [
        {"metric": "documents analysed", "value": duplicates["documents"]},
        {"metric": "exact duplicate groups", "value": duplicates["exact_duplicate_groups"]},
        {"metric": "exact duplicate documents", "value": duplicates["exact_duplicate_documents"]},
        {"metric": "normalized duplicate groups", "value": duplicates["normalized_duplicate_groups"]},
        {"metric": "normalized duplicate documents", "value": duplicates["normalized_duplicate_documents"]},
        {"metric": "near-duplicate pairs", "value": duplicates["near_duplicate_pairs"]},
        {"metric": "near-duplicate threshold", "value": duplicates["near_duplicate_threshold"]},
    ]
)
display(duplicate_frame)

if duplicates["exact_duplicate_samples"]:
    for group in duplicates["exact_duplicate_samples"][:3]:
        print([documents[index].id for index in group])

,metric,value
0,documents analysed,16144.0
1,exact duplicate groups,0.0
2,exact duplicate documents,0.0
3,normalized duplicate groups,0.0
4,normalized duplicate documents,0.0
5,near-duplicate pairs,0.0
6,near-duplicate threshold,0.9


### Reading the duplicates

No exact, normalized, or candidate near-duplicate pair was found. That result is consistent with a corpus built from
templates with independently sampled PII values: the structure repeats, the text does not.

The near-duplicate check is a heuristic. It only compares documents that share their first ten normalized words and
therefore cannot detect a duplicated body with a different opening. Absence of findings bounds the risk; it does not
prove that no semantic duplicate exists.


## 7. Partitions

**What.** Compare `train`, `val`, and `test` on size, identifier overlap, identical text overlap, and document-type
mix.

**Why.** Partitions must be disjoint and comparable. Overlapping identifiers are a direct leak; a different
document-type mix makes validation and test results incomparable with training.


In [14]:
partitions = audit.partition_report(splits)

display(pd.DataFrame([partitions["sizes"]]).T.rename(columns={0: "examples"}))
display(pd.DataFrame(partitions["identifier_overlap"]).T)
print(f"Within-split duplicated texts: {partitions['within_split_duplicate_texts']}")

type_mix = pd.DataFrame(partitions["document_types"]).fillna(0).astype(int)
type_mix["total"] = type_mix.sum(axis=1)
type_mix = type_mix.sort_values("total", ascending=False).drop(columns="total")
display(type_mix)

,examples
train,12915
val,1614
test,1615


,ids,texts
train->val,0,0
train->test,0,0
val->test,0,0


Within-split duplicated texts: {'train': 0, 'val': 0, 'test': 0}


,train,val,test
denuncia_policial,1898,234,223
contrato_laboral,1787,234,244
factura_comercial,1729,232,242
publicacion_red_social,1689,202,184
reclamo_lopdp,1627,208,224
reclamo_bancario,1480,184,176
correo_soporte,1162,136,133
formulario_matricula,924,109,112
historia_clinica,619,75,77


In [15]:
figure, axis = plt.subplots(figsize=(9, 5))
matrix = type_mix.to_numpy()
image = axis.imshow(matrix, aspect="auto", cmap="Blues")
axis.set_xticks(range(len(type_mix.columns)), type_mix.columns)
axis.set_yticks(range(len(type_mix.index)), type_mix.index)
for row in range(matrix.shape[0]):
    for column in range(matrix.shape[1]):
        axis.text(column, row, str(matrix[row, column]), ha="center", va="center", fontsize=8)
axis.set_title("Document types per split")
figure.colorbar(image, ax=axis, label="Documents")
save_figure(figure, "004_split_document_types.png")

Saved datasets/dataset2/reports/figures/004_split_document_types.png


### Reading the partitions

Splits follow an 80/10/10 proportion and share no identifier and no identical text, so there is no direct leakage.
The document-type mix is proportional: every type appears in validation and test in roughly the same share as in
training, which makes the three partitions comparable.

The split is a random partition of documents. Section 8 shows that direct identifiers (document number, email, phone,
bank account, cards, plate, tax identifier, password) never repeat across documents, so the random partition does not
duplicate an identity. Repeated names and categorical attributes do cross partitions, but they describe shared
vocabulary rather than the same person.


## 8. Annotation quality

**What.** Verify that tags are grammatically valid, that entities carry real content, that declared PII values exist
verbatim in the text, and that the BIO tokenization aligns with the raw document.

**Why.** A tag sequence can be structurally valid and still be wrong. The strongest available check compares the
labels with the generator's own declaration of which values it inserted.


In [16]:
bio_problems = audit.bio_violations(examples, vocabulary)
print(f"BIO violations: {dict(bio_problems) or 'none'}")

punctuation = audit.punctuation_entities(examples)
print(f"Entities made only of punctuation: {dict(punctuation) or 'none'}")

grounding = audit.grounding_report(documents)
print(f"Documents whose declaration is not fully grounded in the text: {grounding['documents_with_gaps']}")
print(f"Declared fields without a value: {grounding['declared_but_missing'] or 'none'}")
print(f"Values absent from the text: {grounding['not_in_text'] or 'none'}")

BIO violations: none


Entities made only of punctuation: none


Documents whose declaration is not fully grounded in the text: 0
Declared fields without a value: none
Values absent from the text: none


In [17]:
consistency = audit.declaration_consistency(documents)
print(f"Declaration mismatches: {consistency['mismatches']}")
print(f"Documents with declared missing fields: {consistency['documents_with_missing_fields']}")
display(pd.DataFrame([consistency["coverage_stats"]]).rename(index={0: "declared cobertura_pii (%)"}))

Declaration mismatches: {'declared_without_value': 0, 'value_without_declaration': 0}
Documents with declared missing fields: 0


,count,min,p50,p90,p99,max,mean
declared cobertura_pii (%),16144,100.0,100.0,100.0,100.0,100.0,100.0


In [18]:
entity_lengths = audit.entity_token_lengths(examples)
entity_length_stats = audit.length_stats(entity_lengths)
display(pd.DataFrame([entity_length_stats]).rename(index={0: "entity length in tokens"}))

figure, axis = plt.subplots(figsize=(9, 4.5))
axis.hist(entity_lengths, bins=range(1, 13), align="left", color="#c58b3b", edgecolor="white")
axis.set_xlabel("Tokens per entity")
axis.set_ylabel("Entities")
axis.set_title("Entity length distribution")
axis.grid(axis="y", alpha=0.3)
save_figure(figure, "005_entity_lengths.png")

,count,min,p50,p90,p99,max,mean
entity length in tokens,154142,1.0,2.0,7.0,12.0,14.0,2.762563


Saved datasets/dataset2/reports/figures/005_entity_lengths.png


In [19]:
partition_of = {
    example.id: split_name for split_name, split_examples in splits.items() for example in split_examples
}
reuse_frame = pd.DataFrame(audit.value_reuse_report(documents, partition_of))
reuse_frame = reuse_frame.sort_values("cross_split_values", ascending=False)
display(reuse_frame)

fields_with_cross_split = int((reuse_frame["cross_split_values"] > 0).sum())
print(f"Fields with values reused across splits: {fields_with_cross_split} of {len(reuse_frame)}")
print(f"Largest number of documents sharing one value: {int(reuse_frame['max_documents_per_value'].max())}")

,field,distinct_values,reused_values,cross_split_values,max_documents_per_value
15,nombres,918,918,888,48
21,profesion,473,469,365,16
1,cargo,472,467,356,18
10,empresa,4997,456,167,4
23,salario,1078,211,82,4
9,edad,58,58,50,17
18,peso,46,46,42,21
24,talla_estatura,41,41,38,27
0,apellidos,16019,125,33,2
27,universidad,25,25,25,148


Fields with values reused across splits: 19 of 29
Largest number of documents sharing one value: 673


### Reading the annotation quality

The BIO grammar is clean: no unknown tags, no orphan `I-` tags, and no entity made only of punctuation. Every entity
is a real token span.

Grounding is perfect. For all 16,144 documents, every declared field has a value and every value appears verbatim in
the normalized text; the declared `cobertura_pii` equals 1.0 everywhere and `pii_faltante` is always empty. The labels
therefore agree with the generator's declaration, which is the strongest quality evidence available for this corpus.

Entity spans are short but uneven: the median is two tokens, the 90th percentile is seven, and the longest span has 14
tokens, so names, addresses, and clinical terms coexist with single-token identifiers. Token-level classification is
appropriate, but span-level metrics must not assume short entities.

The reuse table separates two different phenomena. Direct identifiers are unique per document: `CEDULA`,
`CORREO_ELECTRONICO`, `NUMERO_TELEFONICO`, `CUENTA_BANCARIA`, both card types, `PLACA_VEHICULO`, `RUC`, and
`CONTRASENA` show no cross-document reuse. Reuse is concentrated in categorical and descriptive fields
(`NACIONALIDAD` has 5 distinct values, `CARRERA` 8, `ESTADO_CIVIL` 4, `ENFERMEDADES` 6) and in names (918 distinct
first names, reused in up to 48 documents). The corpus therefore does not duplicate identities across splits; the
risk is that frequent categories are over-represented, which class weighting addresses rather than grouped splitting.

Two limits remain. Grounding cannot detect a label that is grounded but attached to the wrong occurrence when the
same value appears several times in a document. And the tokenization warning from Section 3 (1,207 of 16,144
documents) means the BIO token boundaries cannot always be recomputed from `texto`; downstream code must consume
`tokens` as authoritative.


## 9. Coverage

**What.** Quantify how many documents support each class, identify rare classes, and confirm that every class appears
in every split.

**Why.** A class present in training but absent from validation or test cannot be evaluated. A class supported by a
handful of documents will produce unstable metrics and poor generalization, regardless of model size.


In [20]:
def _types(tags):
    """Entity types present in a tag sequence."""
    return {tag[2:] for tag in tags if tag != "O"}


coverage_rows = []
for entity_type in sorted(document_counts, key=lambda name: document_counts[name]):
    coverage_rows.append(
        {
            "entity_type": entity_type,
            "documents": document_counts[entity_type],
            "share_of_documents": f"{document_counts[entity_type] / len(examples):.2%}",
            "train": sum(1 for example in splits["train"] if entity_type in _types(example.tags)),
            "val": sum(1 for example in splits["val"] if entity_type in _types(example.tags)),
            "test": sum(1 for example in splits["test"] if entity_type in _types(example.tags)),
        }
    )

coverage_frame = pd.DataFrame(coverage_rows)
display(coverage_frame)

,entity_type,documents,share_of_documents,train,val,test
0,ENFERMEDADES,477,2.95%,374,55,48
1,EDAD,528,3.27%,422,49,57
2,ESTADO_CIVIL,548,3.39%,436,52,60
3,PESO,604,3.74%,487,57,60
4,TALLA_ESTATURA,702,4.35%,559,71,72
5,NACIONALIDAD,799,4.95%,644,79,76
6,PIN,907,5.62%,734,89,84
7,TARJETA_CREDITO,930,5.76%,744,88,98
8,TARJETA_DEBITO,954,5.91%,764,105,85
9,FECHA_NACIMIENTO,1023,6.34%,824,101,98


The lower tail matters more than the head: classes supported by fewer than 1,000 documents, under 7% of the
corpus, cannot support a reliable per-class claim without weighting and stratification.


In [21]:
rare = coverage_frame[coverage_frame["documents"] < 1000]
missing_in_split = coverage_frame[(coverage_frame[["train", "val", "test"]] == 0).any(axis=1)]

print(f"Classes with fewer than 1,000 supporting documents: {len(rare)}")
display(rare)
print(f"Classes absent from at least one split: {len(missing_in_split)}")
display(missing_in_split)

Classes with fewer than 1,000 supporting documents: 9


,entity_type,documents,share_of_documents,train,val,test
0,ENFERMEDADES,477,2.95%,374,55,48
1,EDAD,528,3.27%,422,49,57
2,ESTADO_CIVIL,548,3.39%,436,52,60
3,PESO,604,3.74%,487,57,60
4,TALLA_ESTATURA,702,4.35%,559,71,72
5,NACIONALIDAD,799,4.95%,644,79,76
6,PIN,907,5.62%,734,89,84
7,TARJETA_CREDITO,930,5.76%,744,88,98
8,TARJETA_DEBITO,954,5.91%,764,105,85


Classes absent from at least one split: 0


,entity_type,documents,share_of_documents,train,val,test


### Reading the coverage

The two absent classes (`GENERO`, `HISTORIAL_CLINICO`) have no labelled evidence and must be treated as out of scope
for v1, not as classes the model "fails" to learn. Every observed class appears in all three splits, so evaluation can
report per-class results without undefined values.

The lower tail is the real constraint: nine classes are supported by fewer than 1,000 documents, and the weakest
(`ENFERMEDADES`) appears in only 477. They require class-weighted training, stratified reporting, and careful
interpretation. Thresholds for "sufficient support" belong to the modeling plan, not to this audit.


## 10. Summary and hand-off

**What.** Consolidate the audited metrics into one table and state the conclusions that preparation must carry
forward.

**Why.** The audit only has value if it changes decisions. This section is the hand-off artifact that the written
report expands with recommendations.


In [22]:
summary_rows = [
    ("files in the DVC version", len(inventory)),
    ("payload size (MiB)", round(inventory_frame["size_mib"].sum(), 2)),
    ("documents", len(documents)),
    ("labelled examples", len(examples)),
    ("document types", len(document_types)),
    ("declared entity types", len(declared_types)),
    ("observed entity types", len(observed_types)),
    ("entities", sum(entity_counts.values())),
    ("validation errors", summary["errors"]),
    ("validation warnings", summary["warnings"]),
    ("exact duplicate documents", duplicates["exact_duplicate_documents"]),
    ("near-duplicate candidate pairs", duplicates["near_duplicate_pairs"]),
    ("documents with ungrounded PII", grounding["documents_with_gaps"]),
    ("fields with cross-split value reuse", fields_with_cross_split),
    ("classes absent from at least one split", len(missing_in_split)),
]
display(pd.DataFrame(summary_rows, columns=["metric", "value"]))

,metric,value
0,files in the DVC version,10.00
1,payload size (MiB),296.73
2,documents,16144.00
3,labelled examples,16144.00
4,document types,9.00
5,declared entity types,31.00
6,observed entity types,29.00
7,entities,154142.00
8,validation errors,0.00
9,validation warnings,1.00


### Conclusions

1. The frozen version is structurally sound. Fields, types, identifiers, token/tag lengths, and BIO grammar pass every
   check; the only warning is a tokenization boundary drift that affects preparation code, not label validity.
2. Annotation quality is high against the strongest available reference: every declared PII value is present in the
   text it labels, and the declared coverage is 1.0 for all documents.
3. The dataset is imbalanced and long-tailed. Two declared classes have no observed evidence, and several classes are
   supported by a few hundred documents, so per-class evaluation is mandatory.
4. No duplicated documents or split leaks were detected. Direct identifiers are unique per document, and the reuse
   observed is limited to categorical attributes and personal names.
5. Documents are long: truncation choices will remove entity spans unless the preparation stage segments text with
   offsets instead of cutting blindly.

### Limitations of this audit

- Duplicate detection is exact plus a prefix-based near-duplicate heuristic; semantic paraphrases are not detected.
- Grounding checks presence, not correct attachment when a value repeats inside one document.
- Label correctness is inferred from the generator declaration; the audit cannot prove that a human annotator would
  label the same spans.
- Coverage thresholds are descriptive; no statistical test of sufficiency was applied.
- The audit covers the frozen `dataset2` version only. Any new DVC version requires a new audit run.

### Next steps

1. Freeze this report as `reports/dataset_audit_v1.md` for the thesis record.
2. Decide the v1 label space: 29 observed classes, with `GENERO` and `HISTORIAL_CLINICO` excluded or re-sampled.
3. Define segmentation and truncation for long documents without losing spans.
4. Persist the split identifiers as a manifest so later dataset versions keep comparable partitions.
5. Only then start the preparation and training phases.
